# SASRec Time-Aware BPI2012 Colab Train (`anchor_ml20` baseline)

Colab notebook for the next time-aware SASRec experiment after choosing `anchor_ml20` as the final baseline.

Goals:
- reuse the completed `anchor_ml20` baseline results
- train only the new time-aware runs
- compare baseline vs `8-bucket` vs `9-bucket`
- evaluate under both `NDCG@10` and `NDCG@5` model-selection criteria


In [1]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.10.0+cu128
cuda available: True
gpu name: Tesla T4


In [24]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
BASELINE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10'
BASELINE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5'
TIMEAWARE_NDCG10_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg10'
TIMEAWARE_NDCG5_OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg5'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DATA_DIR:', DATA_DIR)
print('BASELINE_NDCG10_OUTPUT_DIR:', BASELINE_NDCG10_OUTPUT_DIR)
print('BASELINE_NDCG5_OUTPUT_DIR:', BASELINE_NDCG5_OUTPUT_DIR)
print('TIMEAWARE_NDCG10_OUTPUT_DIR:', TIMEAWARE_NDCG10_OUTPUT_DIR)
print('TIMEAWARE_NDCG5_OUTPUT_DIR:', TIMEAWARE_NDCG5_OUTPUT_DIR)


DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only
BASELINE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg10
BASELINE_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012_ndcg5
TIMEAWARE_NDCG10_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg10
TIMEAWARE_NDCG5_OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg5


In [4]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$BASELINE_NDCG10_OUTPUT_DIR"
!mkdir -p "$BASELINE_NDCG5_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG10_OUTPUT_DIR"
!mkdir -p "$TIMEAWARE_NDCG5_OUTPUT_DIR"


In [5]:
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction
!git pull


/content
/content/time-aware-behavior-prediction
Already up to date.


In [6]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [7]:
!pip install -r requirements_colab.txt


In [8]:
!ls "$DATA_DIR"


events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [9]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


## Experiment design

Fixed baseline setting:
- `anchor_ml20`
- `hidden_units=32, num_blocks=2, num_heads=1, maxlen=20, lr=0.001, dropout=0.2`
- seeds: `42`, `2024`, `7`

Comparison targets:
- baseline (reuse existing completed runs)
- time-aware `8-bucket`
- time-aware `9-bucket`

Time-aware design:
- `x = item_embedding + positional_embedding + time_embedding`
- time source: `delta_prev_seconds`


## Check existing baseline runs

These baseline runs should already exist and must not be retrained.


In [10]:
from pathlib import Path

baseline_ndcg10_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
]
baseline_ndcg5_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
]

for label, output_dir, run_names in [
    ('Baseline NDCG@10', Path(BASELINE_NDCG10_OUTPUT_DIR), baseline_ndcg10_runs),
    ('Baseline NDCG@5', Path(BASELINE_NDCG5_OUTPUT_DIR), baseline_ndcg5_runs),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'MISSING')


Baseline NDCG@10
anchor_ml20_s42 EXISTS
anchor_ml20_s2024 EXISTS
anchor_ml20_s7 EXISTS
Baseline NDCG@5
anchor_ml20_s42 EXISTS
anchor_ml20_s2024 EXISTS
anchor_ml20_s7 EXISTS


## Check planned time-aware runs

Only train runs that are still missing.


In [11]:
planned_ndcg10 = [
    'timeaware_anchor_ml20_b8_s42',
    'timeaware_anchor_ml20_b8_s2024',
    'timeaware_anchor_ml20_b8_s7',
    'timeaware_anchor_ml20_b9_s42',
    'timeaware_anchor_ml20_b9_s2024',
    'timeaware_anchor_ml20_b9_s7',
]
planned_ndcg5 = [
    'timeaware_anchor_ml20_b8_s42',
    'timeaware_anchor_ml20_b8_s2024',
    'timeaware_anchor_ml20_b8_s7',
    'timeaware_anchor_ml20_b9_s42',
    'timeaware_anchor_ml20_b9_s2024',
    'timeaware_anchor_ml20_b9_s7',
]

for label, output_dir, run_names in [
    ('Time-aware NDCG@10', Path(TIMEAWARE_NDCG10_OUTPUT_DIR), planned_ndcg10),
    ('Time-aware NDCG@5', Path(TIMEAWARE_NDCG5_OUTPUT_DIR), planned_ndcg5),
]:
    print('=' * 80)
    print(label)
    for run_name in run_names:
        run_dir = output_dir / run_name
        print(run_name, 'EXISTS' if run_dir.exists() else 'OK')


Time-aware NDCG@10
timeaware_anchor_ml20_b8_s42 OK
timeaware_anchor_ml20_b8_s2024 OK
timeaware_anchor_ml20_b8_s7 OK
timeaware_anchor_ml20_b9_s42 OK
timeaware_anchor_ml20_b9_s2024 OK
timeaware_anchor_ml20_b9_s7 OK
Time-aware NDCG@5
timeaware_anchor_ml20_b8_s42 OK
timeaware_anchor_ml20_b8_s2024 OK
timeaware_anchor_ml20_b8_s7 OK
timeaware_anchor_ml20_b9_s42 OK
timeaware_anchor_ml20_b9_s2024 OK
timeaware_anchor_ml20_b9_s7 OK


## Train time-aware runs for `NDCG@10`

Run these cells only if the corresponding run directory does not already exist.


### timeaware_anchor_ml20_b8_s42


In [12]:
!python src/train_sasrec.py \
  --run_name timeaware_anchor_ml20_b8_s42 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg10/timeaware_anchor_ml20_b8_s42
epoch=1, loss=0.6426
epoch=2, loss=0.3038
epoch=3, loss=0.2099
epoch=4, loss=0.1665
epoch=5, loss=0.1410
valid [full], NDCG@5: 0.6361, HR@5: 0.7458, NDCG@10: 0.7103, HR@10: 0.9801, MRR: 0.6324
valid [sampled], NDCG@5: 0.5426, HR@5: 0.5440, NDCG@10: 0.5502, HR@10: 0.5689, MRR: 0.5602
test [full], NDCG@5: 0.7578, HR@5: 0.8828, NDCG@10: 0.7957, HR@10: 0.9991, MRR: 0.7315
test [sampled], NDCG@5: 0.1689, HR@5: 0.2476, NDCG@10: 0.2490, HR@10: 0.4949, MRR: 0.2017
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware

### timeaware_anchor_ml20_b8_s2024


In [13]:
!python src/train_sasrec.py \
  --run_name timeaware_anchor_ml20_b8_s2024 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg10/timeaware_anchor_ml20_b8_s2024
epoch=1, loss=0.6654
epoch=2, loss=0.3158
epoch=3, loss=0.2140
epoch=4, loss=0.1656
epoch=5, loss=0.1401
valid [full], NDCG@5: 0.6224, HR@5: 0.7353, NDCG@10: 0.7023, HR@10: 0.9882, MRR: 0.6187
valid [sampled], NDCG@5: 0.5199, HR@5: 0.5203, NDCG@10: 0.5285, HR@10: 0.5485, MRR: 0.5395
test [full], NDCG@5: 0.7852, HR@5: 0.8973, NDCG@10: 0.8148, HR@10: 0.9903, MRR: 0.7618
test [sampled], NDCG@5: 0.2319, HR@5: 0.3067, NDCG@10: 0.2923, HR@10: 0.4919, MRR: 0.2577
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-awa

### timeaware_anchor_ml20_b8_s7


In [14]:
!python src/train_sasrec.py \
  --run_name timeaware_anchor_ml20_b8_s7 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg10/timeaware_anchor_ml20_b8_s7
epoch=1, loss=0.6868
epoch=2, loss=0.3246
epoch=3, loss=0.2269
epoch=4, loss=0.1721
epoch=5, loss=0.1438
valid [full], NDCG@5: 0.6513, HR@5: 0.7788, NDCG@10: 0.7040, HR@10: 0.9438, MRR: 0.6369
valid [sampled], NDCG@5: 0.5570, HR@5: 0.5583, NDCG@10: 0.5603, HR@10: 0.5687, MRR: 0.5723
test [full], NDCG@5: 0.6535, HR@5: 0.9260, NDCG@10: 0.6711, HR@10: 0.9776, MRR: 0.5739
test [sampled], NDCG@5: 0.1664, HR@5: 0.1686, NDCG@10: 0.1840, HR@10: 0.2262, MRR: 0.2136
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-

### timeaware_anchor_ml20_b9_s42


In [15]:
!python src/train_sasrec.py \
  --run_name timeaware_anchor_ml20_b9_s42 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg10/timeaware_anchor_ml20_b9_s42
epoch=1, loss=0.6439
epoch=2, loss=0.3050
epoch=3, loss=0.2097
epoch=4, loss=0.1655
epoch=5, loss=0.1402
valid [full], NDCG@5: 0.6423, HR@5: 0.7503, NDCG@10: 0.7151, HR@10: 0.9814, MRR: 0.6386
valid [sampled], NDCG@5: 0.5593, HR@5: 0.5607, NDCG@10: 0.5657, HR@10: 0.5815, MRR: 0.5756
test [full], NDCG@5: 0.7782, HR@5: 0.8932, NDCG@10: 0.8132, HR@10: 1.0000, MRR: 0.7539
test [sampled], NDCG@5: 0.1876, HR@5: 0.2663, NDCG@10: 0.2698, HR@10: 0.5204, MRR: 0.2207
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware

### timeaware_anchor_ml20_b9_s2024


In [16]:
!python src/train_sasrec.py \
  --run_name timeaware_anchor_ml20_b9_s2024 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg10/timeaware_anchor_ml20_b9_s2024
epoch=1, loss=0.6604
epoch=2, loss=0.3156
epoch=3, loss=0.2133
epoch=4, loss=0.1649
epoch=5, loss=0.1394
valid [full], NDCG@5: 0.6306, HR@5: 0.7476, NDCG@10: 0.7033, HR@10: 0.9797, MRR: 0.6233
valid [sampled], NDCG@5: 0.5281, HR@5: 0.5292, NDCG@10: 0.5369, HR@10: 0.5577, MRR: 0.5469
test [full], NDCG@5: 0.7836, HR@5: 0.9071, NDCG@10: 0.8092, HR@10: 0.9884, MRR: 0.7549
test [sampled], NDCG@5: 0.2303, HR@5: 0.3041, NDCG@10: 0.2889, HR@10: 0.4835, MRR: 0.2562
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-awa

### timeaware_anchor_ml20_b9_s7


In [17]:
!python src/train_sasrec.py \
  --run_name timeaware_anchor_ml20_b9_s7 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg10" \
  --selection_metric full_valid_ndcg@10 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg10/timeaware_anchor_ml20_b9_s7
epoch=1, loss=0.6771
epoch=2, loss=0.3217
epoch=3, loss=0.2233
epoch=4, loss=0.1713
epoch=5, loss=0.1432
valid [full], NDCG@5: 0.6560, HR@5: 0.8076, NDCG@10: 0.7039, HR@10: 0.9593, MRR: 0.6304
valid [sampled], NDCG@5: 0.5389, HR@5: 0.5402, NDCG@10: 0.5440, HR@10: 0.5565, MRR: 0.5563
test [full], NDCG@5: 0.6490, HR@5: 0.9041, NDCG@10: 0.6739, HR@10: 0.9796, MRR: 0.5769
test [sampled], NDCG@5: 0.1676, HR@5: 0.1700, NDCG@10: 0.1851, HR@10: 0.2276, MRR: 0.2152
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-

## Train time-aware runs for `NDCG@5`

Run these cells only if the corresponding run directory does not already exist.


### timeaware_anchor_ml20_b8_s42


In [18]:
!python src/train_sasrec.py \
  --run_name timeaware_anchor_ml20_b8_s42 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg5/timeaware_anchor_ml20_b8_s42
epoch=1, loss=0.6426
epoch=2, loss=0.3038
epoch=3, loss=0.2099
epoch=4, loss=0.1665
epoch=5, loss=0.1410
valid [full], NDCG@5: 0.6361, HR@5: 0.7458, NDCG@10: 0.7103, HR@10: 0.9801, MRR: 0.6324
valid [sampled], NDCG@5: 0.5426, HR@5: 0.5440, NDCG@10: 0.5502, HR@10: 0.5689, MRR: 0.5602
test [full], NDCG@5: 0.7578, HR@5: 0.8828, NDCG@10: 0.7957, HR@10: 0.9991, MRR: 0.7315
test [sampled], NDCG@5: 0.1689, HR@5: 0.2476, NDCG@10: 0.2490, HR@10: 0.4949, MRR: 0.2017
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-b

### timeaware_anchor_ml20_b8_s2024


In [19]:
!python src/train_sasrec.py \
  --run_name timeaware_anchor_ml20_b8_s2024 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg5/timeaware_anchor_ml20_b8_s2024
epoch=1, loss=0.6654
epoch=2, loss=0.3158
epoch=3, loss=0.2140
epoch=4, loss=0.1656
epoch=5, loss=0.1401
valid [full], NDCG@5: 0.6224, HR@5: 0.7353, NDCG@10: 0.7023, HR@10: 0.9882, MRR: 0.6187
valid [sampled], NDCG@5: 0.5199, HR@5: 0.5203, NDCG@10: 0.5285, HR@10: 0.5485, MRR: 0.5395
test [full], NDCG@5: 0.7852, HR@5: 0.8973, NDCG@10: 0.8148, HR@10: 0.9903, MRR: 0.7618
test [sampled], NDCG@5: 0.2319, HR@5: 0.3067, NDCG@10: 0.2923, HR@10: 0.4919, MRR: 0.2577
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware

### timeaware_anchor_ml20_b8_s7


In [20]:
!python src/train_sasrec.py \
  --run_name timeaware_anchor_ml20_b8_s7 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg5/timeaware_anchor_ml20_b8_s7
epoch=1, loss=0.6868
epoch=2, loss=0.3246
epoch=3, loss=0.2269
epoch=4, loss=0.1721
epoch=5, loss=0.1438
valid [full], NDCG@5: 0.6513, HR@5: 0.7788, NDCG@10: 0.7040, HR@10: 0.9438, MRR: 0.6369
valid [sampled], NDCG@5: 0.5570, HR@5: 0.5583, NDCG@10: 0.5603, HR@10: 0.5687, MRR: 0.5723
test [full], NDCG@5: 0.6535, HR@5: 0.9260, NDCG@10: 0.6711, HR@10: 0.9776, MRR: 0.5739
test [sampled], NDCG@5: 0.1664, HR@5: 0.1686, NDCG@10: 0.1840, HR@10: 0.2262, MRR: 0.2136
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-be

### timeaware_anchor_ml20_b9_s42


In [21]:
!python src/train_sasrec.py \
  --run_name timeaware_anchor_ml20_b9_s42 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 42 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg5/timeaware_anchor_ml20_b9_s42
epoch=1, loss=0.6439
epoch=2, loss=0.3050
epoch=3, loss=0.2097
epoch=4, loss=0.1655
epoch=5, loss=0.1402
valid [full], NDCG@5: 0.6423, HR@5: 0.7503, NDCG@10: 0.7151, HR@10: 0.9814, MRR: 0.6386
valid [sampled], NDCG@5: 0.5593, HR@5: 0.5607, NDCG@10: 0.5657, HR@10: 0.5815, MRR: 0.5756
test [full], NDCG@5: 0.7782, HR@5: 0.8932, NDCG@10: 0.8132, HR@10: 1.0000, MRR: 0.7539
test [sampled], NDCG@5: 0.1876, HR@5: 0.2663, NDCG@10: 0.2698, HR@10: 0.5204, MRR: 0.2207
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-b

### timeaware_anchor_ml20_b9_s2024


In [22]:
!python src/train_sasrec.py \
  --run_name timeaware_anchor_ml20_b9_s2024 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 2024 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg5/timeaware_anchor_ml20_b9_s2024
epoch=1, loss=0.6604
epoch=2, loss=0.3156
epoch=3, loss=0.2133
epoch=4, loss=0.1649
epoch=5, loss=0.1394
valid [full], NDCG@5: 0.6306, HR@5: 0.7476, NDCG@10: 0.7033, HR@10: 0.9797, MRR: 0.6233
valid [sampled], NDCG@5: 0.5281, HR@5: 0.5292, NDCG@10: 0.5369, HR@10: 0.5577, MRR: 0.5469
test [full], NDCG@5: 0.7836, HR@5: 0.9071, NDCG@10: 0.8092, HR@10: 0.9884, MRR: 0.7549
test [sampled], NDCG@5: 0.2303, HR@5: 0.3041, NDCG@10: 0.2889, HR@10: 0.4835, MRR: 0.2562
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware

### timeaware_anchor_ml20_b9_s7


In [23]:
!python src/train_sasrec.py \
  --run_name timeaware_anchor_ml20_b9_s7 \
  --hidden_units 32 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 20 \
  --lr 0.001 \
  --dropout_rate 0.2 \
  --seed 7 \
  --use_time_embedding \
  --time_delta_column delta_prev_seconds \
  --time_bucket_first_event_separate \
  --time_bucket_zero_gap_separate \
  --time_bucket_boundaries 60,600,3600,86400,604800 \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg5" \
  --selection_metric full_valid_ndcg@5 \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@5
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_timeaware_anchor_ml20_ndcg5/timeaware_anchor_ml20_b9_s7
epoch=1, loss=0.6771
epoch=2, loss=0.3217
epoch=3, loss=0.2233
epoch=4, loss=0.1713
epoch=5, loss=0.1432
valid [full], NDCG@5: 0.6560, HR@5: 0.8076, NDCG@10: 0.7039, HR@10: 0.9593, MRR: 0.6304
valid [sampled], NDCG@5: 0.5389, HR@5: 0.5402, NDCG@10: 0.5440, HR@10: 0.5565, MRR: 0.5563
test [full], NDCG@5: 0.6490, HR@5: 0.9041, NDCG@10: 0.6739, HR@10: 0.9796, MRR: 0.5769
test [sampled], NDCG@5: 0.1676, HR@5: 0.1700, NDCG@10: 0.1851, HR@10: 0.2276, MRR: 0.2152
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-be

## Rebuild result tables from run folders

This avoids schema issues and lets us combine existing baseline runs with new time-aware runs safely.


In [26]:
from pathlib import Path
import json
import pandas as pd

def rebuild_df(output_dir: str):
    rows = []
    output_path = Path(output_dir)
    if not output_path.exists():
        return pd.DataFrame()
    for run_dir in output_path.iterdir():
        if not run_dir.is_dir():
            continue
        summary_path = run_dir / 'metrics_summary.json'
        config_path = run_dir / 'config.json'
        if not summary_path.exists() or not config_path.exists():
            continue
        summary = json.loads(summary_path.read_text(encoding='utf-8'))
        config = json.loads(config_path.read_text(encoding='utf-8'))
        row = {
            'run_name': summary.get('run_name'),
            'run_dir': str(run_dir),
            'completed_at': summary.get('completed_at'),
            'best_epoch': summary.get('best_epoch'),
            'checkpoint_best': summary.get('checkpoint_best'),
            'checkpoint_last': summary.get('checkpoint_last'),
            'metrics_history': summary.get('metrics_history'),
            'config_path': str(config_path),
            'metrics_summary': str(summary_path),
            'maxlen': config.get('maxlen'),
            'dropout_rate': config.get('dropout_rate'),
            'hidden_units': config.get('hidden_units'),
            'seed': config.get('seed'),
            'selection_metric': config.get('selection_metric'),
            'use_time_embedding': config.get('use_time_embedding', False),
            'time_bucket_boundaries': ','.join(str(x) for x in config.get('time_bucket_boundaries', [])),
            'time_bucket_count': config.get('time_bucket_count'),
        }
        best_valid = summary.get('best_valid', {})
        best_test = summary.get('best_test_at_best_valid', {})
        def pick(metrics_group, mode, key):
            return metrics_group.get(mode, {}).get(key)
        row.update({
            'best_valid_full_ndcg@10': pick(best_valid, 'full', 'ndcg@10'),
            'best_valid_full_hr@10': pick(best_valid, 'full', 'hr@10'),
            'best_valid_full_ndcg@5': pick(best_valid, 'full', 'ndcg@5'),
            'best_valid_full_hr@5': pick(best_valid, 'full', 'hr@5'),
            'best_valid_full_mrr': pick(best_valid, 'full', 'mrr'),
            'best_test_full_ndcg@10': pick(best_test, 'full', 'ndcg@10'),
            'best_test_full_hr@10': pick(best_test, 'full', 'hr@10'),
            'best_test_full_ndcg@5': pick(best_test, 'full', 'ndcg@5'),
            'best_test_full_hr@5': pick(best_test, 'full', 'hr@5'),
            'best_test_full_mrr': pick(best_test, 'full', 'mrr'),
            'best_valid_sampled_ndcg@10': pick(best_valid, 'sampled', 'ndcg@10'),
            'best_valid_sampled_hr@10': pick(best_valid, 'sampled', 'hr@10'),
            'best_valid_sampled_ndcg@5': pick(best_valid, 'sampled', 'ndcg@5'),
            'best_valid_sampled_hr@5': pick(best_valid, 'sampled', 'hr@5'),
            'best_valid_sampled_mrr': pick(best_valid, 'sampled', 'mrr'),
            'best_test_sampled_ndcg@10': pick(best_test, 'sampled', 'ndcg@10'),
            'best_test_sampled_hr@10': pick(best_test, 'sampled', 'hr@10'),
            'best_test_sampled_ndcg@5': pick(best_test, 'sampled', 'ndcg@5'),
            'best_test_sampled_hr@5': pick(best_test, 'sampled', 'hr@5'),
            'best_test_sampled_mrr': pick(best_test, 'sampled', 'mrr'),
        })
        rows.append(row)
    return pd.DataFrame(rows)


In [27]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", None)

## NDCG@10 comparison summary


In [28]:
baseline_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
]
timeaware_runs = [
    'timeaware_anchor_ml20_b8_s42',
    'timeaware_anchor_ml20_b8_s2024',
    'timeaware_anchor_ml20_b8_s7',
    'timeaware_anchor_ml20_b9_s42',
    'timeaware_anchor_ml20_b9_s2024',
    'timeaware_anchor_ml20_b9_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG10_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG10_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['bucket_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['bucket_variant'] = timeaware_subset['run_name'].apply(lambda x: 'b8' if '_b8_' in x else 'b9')

df_ndcg10 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg10 = df_ndcg10.sort_values(['bucket_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg10[[
    'run_name', 'seed', 'bucket_variant', 'use_time_embedding', 'time_bucket_boundaries',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,bucket_variant,use_time_embedding,time_bucket_boundaries,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,timeaware_anchor_ml20_b8_s7,7,b8,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.755893,0.958773,0.734980,0.889970,0.695174,0.807914,1.000000,0.807725,0.999458,0.743702,0.607341,0.677989,0.579652,0.590082,0.602256,0.335040,0.452994,0.289917,0.309808,0.335265
1,timeaware_anchor_ml20_b8_s42,42,b8,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.714948,0.976252,0.669355,0.832406,0.636630,0.804902,1.000000,0.794084,0.966734,0.739006,0.537377,0.580867,0.520171,0.526145,0.542696,0.379151,0.515020,0.323546,0.335871,0.371140
2,timeaware_anchor_ml20_b8_s2024,2024,b8,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.729521,0.977380,0.690536,0.855614,0.654781,0.820632,1.000000,0.799047,0.933586,0.762066,0.551496,0.599456,0.533787,0.543810,0.554080,0.412570,0.523190,0.367354,0.377000,0.408648
3,timeaware_anchor_ml20_b9_s7,7,b9,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0,,,6,0,4,8,0,0",0.743844,0.960530,0.714598,0.867397,0.678884,0.843990,1.000000,0.834025,0.971564,0.790520,0.582482,0.666984,0.550773,0.566848,0.573250,0.386793,0.556218,0.322537,0.352208,0.366348
4,timeaware_anchor_ml20_b9_s42,42,b9,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0,,,6,0,4,8,0,0",0.715128,0.981350,0.642260,0.750340,0.638622,0.813155,1.000000,0.778245,0.893199,0.753896,0.565702,0.581509,0.559273,0.560707,0.575554,0.269814,0.520394,0.187604,0.266343,0.220679
5,timeaware_anchor_ml20_b9_s2024,2024,b9,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0,,,6,0,4,8,0,0",0.724989,0.975136,0.655030,0.761685,0.651891,0.810350,1.000000,0.777245,0.897405,0.749042,0.572362,0.618261,0.556768,0.569764,0.571922,0.426353,0.538939,0.380476,0.391281,0.420197
6,anchor_ml20_s7,7,baseline,False,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.753294,0.961857,0.729167,0.883942,0.691225,0.913637,1.000000,0.903861,0.971918,0.885501,0.601240,0.661113,0.580044,0.594609,0.597937,0.480200,0.667438,0.420861,0.484209,0.443705
7,anchor_ml20_s42,42,baseline,False,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.750367,0.977546,0.708784,0.848505,0.681799,0.915296,1.000000,0.913814,0.995781,0.886842,0.583692,0.654340,0.556357,0.567822,0.578433,0.464891,0.652470,0.406641,0.473262,0.430220
8,anchor_ml20_s2024,2024,baseline,False,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.736821,0.967429,0.700301,0.850202,0.668534,0.840611,0.999865,0.819421,0.937829,0.789072,0.574314,0.616064,0.557801,0.563197,0.578608,0.323103,0.518524,0.261120,0.326822,0.292464


In [29]:
summary_ndcg10 = df_ndcg10.groupby('bucket_variant')[[
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg10


best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_mrr           best_test_full_ndcg@10           best_test_full_hr@10           best_test_full_ndcg@5           best_test_full_hr@5           best_test_full_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_mrr           best_test_sampled_ndcg@10           best_test_sampled_hr@10           best_test_sampled_ndcg@5           best_test_sampled_hr@5           best_test_sampled_mrr          
                                  mean       std                  mean       std                   mean       std                 mean       std                mean       std                   mean       std                 mean       std                  mean       std                mean       std               mean       std                       mean       std                     mean       std                      mean       std                    mean       std                   mean       std                      mean       std                    mean       std                     mean       std                   mean       std                  mean       std
bucket_variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
b8                            0.733454  0.020754              0.970802  0.010433               0.698290  0.033493             0.859330  0.028961            0.662195  0.029968               0.811149  0.008349             1.000000  0.000000              0.800285  0.006904            0.966593  0.032936           0.748258  0.012186                   0.565405  0.036998                 0.619437  0.051552                  0.544537  0.031164                0.553345  0.033018               0.566344  0.031618                  0.375587  0.038888                0.497068  0.038387                 0.326939  0.038830               0.340893  0.033877              0.371684  0.036695
b9                            0.727987  0.014591              0.972339  0.010688               0.670629  0.038609             0.793141  0.064557            0.656466  0.020517               0.822498  0.018665             1.000000  0.000000              0.796505  0.032497            0.920722  0.044080           0.764486  0.022676                   0.573515  0.008449                 0.622251  0.042877                  0.555605  0.004368                0.565773  0.004623               0.573575  0.001837                  0.360987  0.081398                0.538517  0.017915                 0.296873  0.098964               0.336610  0.063913              0.335741  0.103220
baseline                      0.746827  0.008788              0.968944  0.007954               0.712751  0.014836             0.860883  0.019988            0.680520  0.011400               0.889848  0.042649             0.999955  0.000078              0.879032  0.051864            0.968510  0.029126           0.853805  0.056064                   0.586415  0.013668                 0.643839  0.024291                  0.564734  0.013279                0.575209  0.016959               0.584993  0.011211                  0.422731  0.086620                0.612810  0.081997                 0.362874  0.088408               0.428098  0.0878

Interpretation guide for NDCG@10:
- first compare `best_valid_full_ndcg@10` and `best_test_full_ndcg@10` across `baseline`, `b8`, and `b9`
- then check whether sampled metrics and MRR show a similar trend
- baseline is reused from the completed sanity-check winner; only time-aware runs are newly trained here


## NDCG@5 comparison summary


In [30]:
baseline_runs = [
    'anchor_ml20_s42',
    'anchor_ml20_s2024',
    'anchor_ml20_s7',
]
timeaware_runs = [
    'timeaware_anchor_ml20_b8_s42',
    'timeaware_anchor_ml20_b8_s2024',
    'timeaware_anchor_ml20_b8_s7',
    'timeaware_anchor_ml20_b9_s42',
    'timeaware_anchor_ml20_b9_s2024',
    'timeaware_anchor_ml20_b9_s7',
]

baseline_df = rebuild_df(BASELINE_NDCG5_OUTPUT_DIR)
timeaware_df = rebuild_df(TIMEAWARE_NDCG5_OUTPUT_DIR)

baseline_subset = baseline_df[baseline_df['run_name'].isin(baseline_runs)].copy()
baseline_subset['model_variant'] = 'baseline'
baseline_subset['bucket_variant'] = 'baseline'

timeaware_subset = timeaware_df[timeaware_df['run_name'].isin(timeaware_runs)].copy()
timeaware_subset['model_variant'] = 'timeaware'
timeaware_subset['bucket_variant'] = timeaware_subset['run_name'].apply(lambda x: 'b8' if '_b8_' in x else 'b9')

df_ndcg5 = pd.concat([baseline_subset, timeaware_subset], ignore_index=True)
df_ndcg5 = df_ndcg5.sort_values(['bucket_variant', 'seed', 'run_name']).reset_index(drop=True)
df_ndcg5[[
    'run_name', 'seed', 'bucket_variant', 'use_time_embedding', 'time_bucket_boundaries',
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]]


,run_name,seed,bucket_variant,use_time_embedding,time_bucket_boundaries,best_valid_full_ndcg@10,best_valid_full_hr@10,best_valid_full_ndcg@5,best_valid_full_hr@5,best_valid_full_mrr,best_test_full_ndcg@10,best_test_full_hr@10,best_test_full_ndcg@5,best_test_full_hr@5,best_test_full_mrr,best_valid_sampled_ndcg@10,best_valid_sampled_hr@10,best_valid_sampled_ndcg@5,best_valid_sampled_hr@5,best_valid_sampled_mrr,best_test_sampled_ndcg@10,best_test_sampled_hr@10,best_test_sampled_ndcg@5,best_test_sampled_hr@5,best_test_sampled_mrr
0,timeaware_anchor_ml20_b8_s7,7,b8,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.755893,0.958773,0.734980,0.889970,0.695174,0.807914,1.000000,0.807725,0.999458,0.743702,0.607341,0.677989,0.579652,0.590082,0.602256,0.335040,0.452994,0.289917,0.309808,0.335265
1,timeaware_anchor_ml20_b8_s42,42,b8,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.714948,0.976252,0.669355,0.832406,0.636630,0.804902,1.000000,0.794084,0.966734,0.739006,0.537377,0.580867,0.520171,0.526145,0.542696,0.379151,0.515020,0.323546,0.335871,0.371140
2,timeaware_anchor_ml20_b8_s2024,2024,b8,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.729521,0.977380,0.690536,0.855614,0.654781,0.820632,1.000000,0.799047,0.933586,0.762066,0.551496,0.599456,0.533787,0.543810,0.554080,0.412570,0.523190,0.367354,0.377000,0.408648
3,timeaware_anchor_ml20_b9_s7,7,b9,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0,,,6,0,4,8,0,0",0.743844,0.960530,0.714598,0.867397,0.678884,0.843990,1.000000,0.834025,0.971564,0.790520,0.582482,0.666984,0.550773,0.566848,0.573250,0.386793,0.556218,0.322537,0.352208,0.366348
4,timeaware_anchor_ml20_b9_s42,42,b9,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0,,,6,0,4,8,0,0",0.710988,0.950629,0.671512,0.825916,0.639986,0.799224,1.000000,0.799134,0.999728,0.731606,0.519510,0.591294,0.492885,0.507642,0.516172,0.398216,0.527555,0.345916,0.359777,0.392595
5,timeaware_anchor_ml20_b9_s2024,2024,b9,True,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0,,,6,0,4,8,0,0",0.721180,0.961533,0.675432,0.821482,0.650350,0.821910,1.000000,0.798775,0.928852,0.763670,0.554774,0.600272,0.538369,0.548707,0.556652,0.422625,0.533089,0.377200,0.386222,0.418122
6,anchor_ml20_s7,7,baseline,False,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.753294,0.961857,0.729167,0.883942,0.691225,0.913637,1.000000,0.903861,0.971918,0.885501,0.601240,0.661113,0.580044,0.594609,0.597937,0.480200,0.667438,0.420861,0.484209,0.443705
7,anchor_ml20_s42,42,baseline,False,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.750367,0.977546,0.708784,0.848505,0.681799,0.915296,1.000000,0.913814,0.995781,0.886842,0.583692,0.654340,0.556357,0.567822,0.578433,0.464891,0.652470,0.406641,0.473262,0.430220
8,anchor_ml20_s2024,2024,baseline,False,"6,0,,,6,0,0,,,3,6,0,0,,,8,6,4,0,0",0.736821,0.967429,0.700301,0.850202,0.668534,0.840611,0.999865,0.819421,0.937829,0.789072,0.574314,0.616064,0.557801,0.563197,0.578608,0.323103,0.518524,0.261120,0.326822,0.292464


In [31]:
summary_ndcg5 = df_ndcg5.groupby('bucket_variant')[[
    'best_valid_full_ndcg@10', 'best_valid_full_hr@10',
    'best_valid_full_ndcg@5', 'best_valid_full_hr@5',
    'best_valid_full_mrr',
    'best_test_full_ndcg@10', 'best_test_full_hr@10',
    'best_test_full_ndcg@5', 'best_test_full_hr@5',
    'best_test_full_mrr',
    'best_valid_sampled_ndcg@10', 'best_valid_sampled_hr@10',
    'best_valid_sampled_ndcg@5', 'best_valid_sampled_hr@5',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@10', 'best_test_sampled_hr@10',
    'best_test_sampled_ndcg@5', 'best_test_sampled_hr@5',
    'best_test_sampled_mrr',
]].agg(['mean', 'std'])
summary_ndcg5


best_valid_full_ndcg@10           best_valid_full_hr@10           best_valid_full_ndcg@5           best_valid_full_hr@5           best_valid_full_mrr           best_test_full_ndcg@10           best_test_full_hr@10           best_test_full_ndcg@5           best_test_full_hr@5           best_test_full_mrr           best_valid_sampled_ndcg@10           best_valid_sampled_hr@10           best_valid_sampled_ndcg@5           best_valid_sampled_hr@5           best_valid_sampled_mrr           best_test_sampled_ndcg@10           best_test_sampled_hr@10           best_test_sampled_ndcg@5           best_test_sampled_hr@5           best_test_sampled_mrr          
                                  mean       std                  mean       std                   mean       std                 mean       std                mean       std                   mean       std                 mean       std                  mean       std                mean       std               mean       std                       mean       std                     mean       std                      mean       std                    mean       std                   mean       std                      mean       std                    mean       std                     mean       std                   mean       std                  mean       std
bucket_variant                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    
b8                            0.733454  0.020754              0.970802  0.010433               0.698290  0.033493             0.859330  0.028961            0.662195  0.029968               0.811149  0.008349             1.000000  0.000000              0.800285  0.006904            0.966593  0.032936           0.748258  0.012186                   0.565405  0.036998                 0.619437  0.051552                  0.544537  0.031164                0.553345  0.033018               0.566344  0.031618                  0.375587  0.038888                0.497068  0.038387                 0.326939  0.038830               0.340893  0.033877              0.371684  0.036695
b9                            0.725337  0.016818              0.957564  0.006027               0.687181  0.023825             0.838265  0.025326            0.656407  0.020144               0.821708  0.022384             1.000000  0.000000              0.810645  0.020249            0.966714  0.035686           0.761932  0.029495                   0.552255  0.031561                 0.619516  0.041352                  0.527343  0.030478                0.541066  0.030334               0.548692  0.029360                  0.402545  0.018304                0.538954  0.015205                 0.348551  0.027427               0.366069  0.017859              0.392355  0.025888
baseline                      0.746827  0.008788              0.968944  0.007954               0.712751  0.014836             0.860883  0.019988            0.680520  0.011400               0.889848  0.042649             0.999955  0.000078              0.879032  0.051864            0.968510  0.029126           0.853805  0.056064                   0.586415  0.013668                 0.643839  0.024291                  0.564734  0.013279                0.575209  0.016959               0.584993  0.011211                  0.422731  0.086620                0.612810  0.081997                 0.362874  0.088408               0.428098  0.0878

Interpretation guide for NDCG@5:
- first compare `best_valid_full_ndcg@5` and `best_test_full_ndcg@5` across `baseline`, `b8`, and `b9`
- then check whether sampled metrics and MRR show a similar trend
- baseline is reused from the completed sanity-check winner; only time-aware runs are newly trained here
